In [1]:
class Expr:
    def __call__(self, **context):
        raise NotImplementedError("Subclasses must implement __call__.")

    def d(self, wrt):
        raise NotImplementedError("Subclasses must implement d.")
    def __add__(self, other):
        if not isinstance(other, Expr):
            other = Const(other)
        return Sum(self, other)

    def __sub__(self, other):
        if not isinstance(other, Expr):
            other = Const(other)
        return Sum(self, -other)

    def __mul__(self, other):
        if not isinstance(other, Expr):
            other = Const(other)
        return Product(self, other)

    def __truediv__(self, other):
        if not isinstance(other, Expr):
            other = Const(other)
        return Fraction(self, other)

    def __neg__(self):
        return Product(Const(-1), self)

In [2]:
class Const(Expr):
    def __init__(self, value):
        self.value = value

    def __call__(self, **context):
        return self.value

    def d(self, wrt):
        return Const(0)

class Var(Expr):
    def __init__(self, name):
        self.name = name

    def __call__(self, **context):
        return context.get(self.name, 0)

    def d(self, wrt):
        return Const(1) if self.name == wrt.name else Const(0)


In [3]:
V = Var
C = Const

class BinOp(Expr):
    def __init__(self, expr1, expr2):
        self.expr1, self.expr2 = expr1, expr2

class Sum(BinOp):
    def __call__(self, **context):
        return self.expr1(**context) + self.expr2(**context)

    def d(self, wrt):
        return Sum(self.expr1.d(wrt), self.expr2.d(wrt))

class Product(BinOp):
    def __call__(self, **context):
        return self.expr1(**context) * self.expr2(**context)

    def d(self, wrt):
        return Sum(
            Product(self.expr1.d(wrt), self.expr2),
            Product(self.expr1, self.expr2.d(wrt)),
        )

class Fraction(BinOp):
    def __call__(self, **context):
        return self.expr1(**context) / self.expr2(**context)

    def d(self, wrt):
        numerator = Sum(
            Product(self.expr1.d(wrt), self.expr2),
            Product(Const(-1), Product(self.expr1, self.expr2.d(wrt))),
        )
        denominator = Product(self.expr2, self.expr2)
        return Fraction(numerator, denominator)

In [4]:
def newton_raphson(expr, x0, eps=1e-4):
    x = V("x")
    f_prime = expr.d(x)
    xn = x0
    while True:
        fxn = expr(x=xn)
        fpxn = f_prime(x=xn)
        if fpxn == 0:
            raise ValueError("Derivative is zero. No solution found.")
        xn1 = xn - fxn / fpxn
        if abs(xn1 - xn) <= eps:
            return xn1
        xn = xn1

In [5]:
print(C(5)())  
print(C(5).d(V("x"))())  
print(V("x")(x=5)) 
print(V("x").d(V("y"))(x=5))  
print(V("x").d(V("x"))(x=5))  

5
0
5
0
1


In [6]:
print(Sum(V("x"), Fraction(V("x"), V("y")))(x=5, y=2.5))  # Output: 7.0
print(Fraction(Sum(C(5), V("y")), Product(V("x"), V("y")))(x=1, y=2))  # Output: 3.5
print(Fraction(Sum(C(5), V("y")), Product(V("x"), V("y"))).d(V("x"))(x=1, y=2))  # Output: -3.5
print(Fraction(Sum(C(5), V("y")), Product(V("x"), V("y"))).d(V("y"))(x=1, y=2))  # Output: -1.25

7.0
3.5
-3.5
-1.25


In [7]:
x = V("x")
f = C(-5) * x * x * x * x * x + C(3) * x + C(2)
zero = newton_raphson(f, 0.5, eps=1e-4)
print(zero, f(x=zero))

1.0000000000000653 -1.4384049507043528e-12


In [8]:
print((V("x") * V("x") / V("y"))(x=5, y=2.5)) 

10.0
